# ECGrecover Baseline Replication

We replicated the ECGrecover paper using the PTB-XL dataset with exact preprocessing from the original paper, with permission from our TAs to use AI to set up the configurations and resolve data formatting issues for training. All code has been reviewed and edited by us. 

**Paper**: [ECGrecover: A Deep Learning Approach for Electrocardiogram Signal Completion](https://arxiv.org/abs/2406.16901)

**Preprocessing (from Section 3.1.2)**:
1. Load 500Hz data (5000 points)
2. Min-max normalize to [-1, 1]
3. Bandpass filter (0.05-150Hz)
4. Downsample to 50Hz (512 points)

---

## Check GPU & Runtime

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Available: {gpu_name}")
    print(f"Memory: {gpu_memory} GB")

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/ECGrecover'
os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)

## Clone ECGrecover Repository

In [ ]:
!git clone https://github.com/UMMISCO/ecgrecover.git
%cd ecgrecover
!ls -la

## Install Dependencies

In [ ]:
!pip install wfdb neurokit2 --quiet

import numpy as np
import scipy
import wfdb

## Fix PyTorch Compatibility Issue

The original code uses `verbose=True` in ReduceLROnPlateau which was removed in newer PyTorch versions so we need to remove it.

In [ ]:
# fix the verbose parameter issue in Training.py
!sed -i 's/verbose=True//' learn/Training.py

## Set PTB-XL Dataset Path

We uploaded the PTB-XL dataset to Google Drive. This allows us to load data from the drive.

The dataset folder contains:
- `records500/` (we need the 500Hz version)
- `ptbxl_database.csv`

In [ ]:
PTBXL_PATH = '/content/drive/MyDrive/Colab Notebooks/ptbxl'

# verify the dataset exists
import os
if os.path.exists(PTBXL_PATH):
    print(f"PTB-XL at {PTBXL_PATH}")
    !ls "{PTBXL_PATH}"

    # check for records500
    if os.path.exists(os.path.join(PTBXL_PATH, 'records500')):
        print("\nrecords500")
    else:
        print("\nERROR")

!ls /content/drive/MyDrive/

## Load PTB-XL at 500Hz

The paper uses 500Hz data, then downsamples to 50Hz after filtering.

In [ ]:
import os
import numpy as np
import pandas as pd
import wfdb
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

def load_single_ecg(args):
    """load a single ECG file"""
    idx, filepath = args
    try:
        signal, meta = wfdb.rdsamp(filepath)
        signal = signal.T.astype(np.float32)
        if signal.shape == (12, 5000):
            return idx, signal
    except:
        pass
    return idx, None

def load_ptbxl_500hz_fast(data_path, num_workers=8):
    """
    load PTB-XL dataset at 500Hz
    """
    Y = pd.read_csv(os.path.join(data_path, 'ptbxl_database.csv'), index_col='ecg_id')

    # prepare file paths
    file_args = [(idx, os.path.join(data_path, row['filename_hr']))
                 for idx, row in Y.iterrows()]

    print(f"loading {len(Y)} ECG records at 500Hz")

    signals = {}
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(load_single_ecg, arg): arg[0] for arg in file_args}

        for future in tqdm(as_completed(futures), total=len(futures)):
            idx, signal = future.result()
            if signal is not None:
                signals[idx] = signal

    # sort by index and stack
    valid_indices = sorted(signals.keys())
    X = np.stack([signals[i] for i in valid_indices], axis=0)
    Y_filtered = Y.loc[valid_indices]

    print(f"\nloaded {len(X)} signals, shape: {X.shape}, range: [{X.min()}, {X.max()}]")

    return X, Y_filtered

X_raw, Y = load_ptbxl_500hz_fast(PTBXL_PATH, num_workers=16)

In [ ]:
# save raw 500Hz data to google drive
import os

raw_data_path = f'{DRIVE_PROJECT_PATH}/data_raw_500hz'
os.makedirs(raw_data_path, exist_ok=True)

np.save(f'{raw_data_path}/X_raw.npy', X_raw)
Y.to_csv(f'{raw_data_path}/Y_metadata.csv')

!ls -lh "{raw_data_path}"

## Preprocess Like the Paper

From Section 3.1.2:
> "We first min-max normalized the ECG data to a range of [-1,1]. We followed a strategy close to [15] to prepare our data: bandpass filtering was applied to the normalized data, using lower and upper cut-off frequencies of 0.05Hz and 150Hz, respectively, to eliminate noise, such as baseline drift and improve signal quality. Finally, we downsampled the original signal from 5000pts (500Hz) to 512pts (50Hz)."

In [ ]:
# from scipy.signal import butter, filtfilt, resample
# from tqdm import tqdm

# def preprocess_ecgrecover(X):
#     n_samples = X.shape[0]
#     target_points = 512  # 50Hz * ~10.24s

#     X_normalized = np.zeros_like(X)
#     for i in tqdm(range(n_samples)):
#         min_val = X[i].min()
#         max_val = X[i].max()
#         if max_val - min_val > 1e-8:
#             X_normalized[i] = 2 * (X[i] - min_val) / (max_val - min_val) - 1
#         else:
#             X_normalized[i] = np.zeros_like(X[i])

#     nyquist = 500 / 2  # 250Hz
#     low = 0.05 / nyquist
#     high = 150 / nyquist
#     b, a = butter(2, [low, high], btype='band')

#     X_filtered = np.zeros_like(X_normalized)
#     for i in tqdm(range(n_samples)):
#         for lead in range(12):
#             X_filtered[i, lead] = filtfilt(b, a, X_normalized[i, lead])

#     X_resampled = np.zeros((n_samples, 12, target_points), dtype=np.float32)
#     for i in tqdm(range(n_samples)):
#         for lead in range(12):
#             X_resampled[i, lead] = resample(X_filtered[i, lead], target_points)

#     for i in range(n_samples):
#         min_val = X_resampled[i].min()
#         max_val = X_resampled[i].max()
#         if max_val - min_val > 1e-8:
#             X_resampled[i] = 2 * (X_resampled[i] - min_val) / (max_val - min_val) - 1

#     print(f"Final shape: {X_resampled.shape}")
#     print(f"Final range: [{X_resampled.min()}, {X_resampled.max()}]")

#     return X_resampled

# X_processed = preprocess_ecgrecover(X_raw)

## Create Train/Val/Test Splits

Following PTB-XL's recommended folds:
- Folds 1-8: Training
- Fold 9: Validation
- Fold 10: Test

In [ ]:
def create_splits(X, Y, output_path):
    """
    Create train/val/test splits using PTB-XL's stratified folds
    """
    train_mask = Y['strat_fold'].isin([1, 2, 3, 4, 5, 6, 7, 8]).values
    val_mask = (Y['strat_fold'] == 9).values
    test_mask = (Y['strat_fold'] == 10).values

    X_train = X[train_mask]
    X_val = X[val_mask]
    X_test = X[test_mask]

    print(f"Train: {X_train.shape}")
    print(f"Val: {X_val.shape}")
    print(f"Test: {X_test.shape}")

    os.makedirs(output_path, exist_ok=True)
    np.save(f'{output_path}/train.npy', X_train.astype(np.float32))
    np.save(f'{output_path}/validation.npy', X_val.astype(np.float32))
    np.save(f'{output_path}/test.npy', X_test.astype(np.float32))

    return X_train, X_val, X_test

X_train, X_val, X_test = create_splits(X_processed, Y, 'data')

In [ ]:
import os
import numpy as np
import pandas as pd
import wfdb
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# load your saved raw 500Hz data
raw_data_path = f'{DRIVE_PROJECT_PATH}/data_raw_500hz'
X_raw = np.load(f'{raw_data_path}/X_raw.npy')
Y = pd.read_csv(f'{raw_data_path}/Y_metadata.csv', index_col='ecg_id')

print(f"Shape: {X_raw.shape}") 

# split into train/val/test (WITHOUT preprocessing)
def create_splits_raw(X, Y, output_path):
    train_mask = Y['strat_fold'].isin([1,2,3,4,5,6,7,8]).values
    val_mask = (Y['strat_fold'] == 9).values
    test_mask = (Y['strat_fold'] == 10).values

    # transpose from (N, 12, 5000) to (N, 5000, 12)
    X_train = X[train_mask].transpose(0, 2, 1)
    X_val = X[val_mask].transpose(0, 2, 1)
    X_test = X[test_mask].transpose(0, 2, 1)

    print(f"Train: {X_train.shape}")  
    print(f"Val: {X_val.shape}")
    print(f"Test: {X_test.shape}")

    os.makedirs(output_path, exist_ok=True)
    np.save(f'{output_path}/train.npy', X_train.astype(np.float32))
    np.save(f'{output_path}/validation.npy', X_val.astype(np.float32))
    np.save(f'{output_path}/test.npy', X_test.astype(np.float32))

    print(f"\nsaved to {output_path}/")

create_splits_raw(X_raw, Y, 'data')

## Verify Preprocessed Data

In [ ]:
train = np.load('data/train.npy')
print(f"Train shape: {train.shape}") 
print(f"Min: {train.min()}")     
print(f"Max: {train.max()}")  
print(f"Mean: {train.mean()}")
print(f"Std: {train.std()}")

!ls -lh data/

## Training Configuration

From the paper:
- 100 epochs --> (we ended up running for 30 epochs after seeing plateaus and signs of overfitting)
- Batch size 256
- Adam optimizer, lr=0.01
- α=0.1 for loss function

In [ ]:
# configuration
EPOCHS = 30
BATCH_SIZE = 256
SEED = 42
GPU_DEVICE = 0

## Train the Model

Command format: `python main.py data_path save_path seed device --Train save_results epochs batch_size`

In [ ]:
# clear any previous model files
!rm -f model/*.pth model/*.npy

# start training
!python main.py data/ model/ {SEED} {GPU_DEVICE} --Train empty {EPOCHS} {BATCH_SIZE}

## Save Model to Google Drive

In [ ]:
import shutil

!ls -la model/

# copy to Drive
if os.path.exists('/content/drive/MyDrive'):
    drive_model_path = f'{DRIVE_PROJECT_PATH}/model'
    os.makedirs(drive_model_path, exist_ok=True)

    for f in os.listdir('model'):
        src = os.path.join('model', f)
        dst = os.path.join(drive_model_path, f)
        shutil.copy2(src, dst)

    print(f"\nsaved to {drive_model_path}")

## Test the Model

In [ ]:
!mv model/Modeltemp.pth model/Model.pth
!python main.py data/test.npy model/ 42 0 Results/

In [ ]:
# view results
!ls -la Results/

In [ ]:
# summary statistics by configuration
print("\n=== Per-Configuration Results ===\n")
summary = metrics.groupby('Config').agg({
    'PCC': ['mean', 'std'],
    'RMSE': ['mean', 'std'],
    'DTW': ['mean', 'std']
}).round(3)
print(summary)

print(f"PCC:  {metrics['PCC'].mean():.3f} ± {metrics['PCC'].std():.3f}")
print(f"RMSE: {metrics['RMSE'].mean():.3f} ± {metrics['RMSE'].std():.3f}")
print(f"DTW:  {metrics['DTW'].mean():.3f} ± {metrics['DTW'].std():.3f}")

In [ ]:
# display CSV results
result_csvs = glob.glob('Results/*.csv')
if result_csvs:
    for csv_path in result_csvs:
        print(f"\n{csv_path}:")
        df = pd.read_csv(csv_path)
        display(df)

## Save Results to Google Drive

In [ ]:
if os.path.exists('/content/drive/MyDrive'):
    drive_results_path = f'{DRIVE_PROJECT_PATH}/Results'
    if os.path.exists('Results') and os.listdir('Results'):
        shutil.copytree('Results', drive_results_path, dirs_exist_ok=True)
        print(f"saved to {drive_results_path}")